# Пригодность данных Ankerprognose для прогнозирования

Временная ось — **erstes Ladedatum**. Единица анализа — уникальный случай
`BWPKKD + Ankerbeginn` в когорте `Ankerjahr × Sachverhalt`.
Исследуем приросты и накопленные Fälle и Lückentage на регулярной недельной сетке.

Notebook читает локальную DuckDB без изменений. Прогнозы и записи в forecast_runs не создаются.
Выходные таблицы агрегированы и не содержат идентификаторов случаев.
Для запуска выберите Python-окружение проекта с pandas, numpy, duckdb и matplotlib/seaborn,
а также установленным ipykernel для Jupyter/VS Code. Выполните Run All.


In [ ]:
from pathlib import Path
import sys
import hashlib
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid', palette='deep')

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "pack" / "data_access.py").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError("Откройте notebook из папки проекта или notebooks.")
sys.path.insert(0, str(ROOT))
from pack.config import ANALYSIS_START_YEAR, DB_PATH

# Параметры исследования. None выбирает последний доступный год с 2024.
# "all" объединяет актуальные когорты, как «Alle ab 2024» в приложении.
ANCHOR_YEAR = None
SACHVERHALT = "all"
HORIZON_WEEKS = 13
MIN_TRAIN_WEEKS = 52
# True допустимо только после подтверждения полноты выгрузки.
CONFIRMED_COMPLETE_EXTRACT = False
# Пользователь подтвердил: недели без новых случаев — обычные нулевые приросты.
# Это не подтверждает полноту выгрузки HANA.
EMPTY_WEEKS_ARE_ZERO = True
ANALYSIS_START = pd.Timestamp(ANALYSIS_START_YEAR, 1, 1)
# Подтвержденная дата полноты данных по erstes Ladedatum, например "2026-11-09".
# Максимальная дата записи сама по себе полноту выгрузки не доказывает.
OBSERVATION_END = None
SHOW_PLOTS = True

def show(fig):
    if SHOW_PLOTS:
        fig.tight_layout()
        plt.show()
    plt.close(fig)

database = Path(DB_PATH)
if not database.is_file():
    raise FileNotFoundError(f"Локальная база отсутствует: {database}")
database_hash_before = hashlib.sha256(database.read_bytes()).hexdigest()
con = duckdb.connect(str(database), read_only=True)
try:
    raw = con.execute('''
        SELECT
            CAST("/B99/S_BWPKKD" AS VARCHAR) AS case_id,
            CAST("/BIC/YBWRSV" AS VARCHAR) AS sachverhalt,
            CAST("/BIC/YBWRSEL" AS VARCHAR) AS selection,
            CAST("/BIC/YBWRIQIQ" AS VARCHAR) AS iq,
            CAST("/BIC/YBWRAVON" AS VARCHAR) AS anchor_start_raw,
            CAST("/BIC/YBWRABIS" AS VARCHAR) AS anchor_end_raw,
            CAST("/BIC/YBWRELAD" AS VARCHAR) AS load_raw,
            CAST("/BIC/YBWRIPL" AS VARCHAR) AS ipl_raw,
            TRY_CAST("/BIC/YBWRFTAGE" AS DOUBLE) AS days
        FROM raw_bestand
    ''').df()
finally:
    con.close()

if raw.empty:
    raise RuntimeError(
        "raw_bestand пуста. Сначала загрузите данные через приложение."
    )

display(pd.Series({
    "Строк": len(raw),
    "Размер базы, MiB": round(database.stat().st_size / 1024**2, 2),
    "Время исследования": str(pd.Timestamp.now()),
}, name="Источник").to_frame())

## 1. Полнота и качество

Отбираем RSA / 4.15.1, как в импорте приложения. Неподходящие строки учитываются
в отчёте. Даты YYYYMMDD, YYYY-MM-DD и DD.MM.YYYY разбираются явно.
IPL используется для выбора последнего состояния случая и проверки снимков.
Ось временных рядов строится по erstes Ladedatum.

Проверяем данные, нужные для анализа и прогноза с 2024 года, включая ранние
снимки актуальных случаев. Полностью старые случаи исключаются из отчёта,
но исходная база не меняется. Нечитаемые даты остаются для проверки.

Ankerende = 31.12.9999 — корректный открытый период. Даты окончания хранятся
как календарные даты Python, чтобы ограничение диапазона pandas Timestamp
не исключало открытые периоды. Они участвуют в детализации и отборе по
пересечению с Ankerjahr; их количество выводится отдельно как информация.


In [ ]:
from datetime import date
from pack.services.quality_service import parse_source_date

def parse_dates(values):
    text = values.astype("string").str.strip()
    dotted = text.str.fullmatch(r"\d{2}\.\d{2}\.\d{4}", na=False)
    parsed = pd.to_datetime(text.where(~dotted), format="mixed", errors="coerce")
    parsed.loc[dotted] = pd.to_datetime(
        text.loc[dotted], format="%d.%m.%Y", errors="coerce"
    )
    return parsed

scope = raw["selection"].eq("4.15.1") & raw["iq"].eq("RSA")
display(pd.Series({"Все строки": len(raw), "RSA / 4.15.1": int(scope.sum()),
                   "Вне выборки": int((~scope).sum())}).to_frame("Строк"))
data = raw.loc[scope].copy()
if data.empty:
    raise RuntimeError("Нет записей RSA / 4.15.1.")
for target, source in [
    ("anchor_start", "anchor_start_raw"), ("anchor_end", "anchor_end_raw"),
    ("load_date", "load_raw"), ("ipl_date", "ipl_raw"),
]:
    if target == "anchor_end":
        # Сохраняем реальную дату 31.12.9999 без замены условной датой.
        data[target] = pd.Series(
            data[source].map(parse_source_date), index=data.index, dtype=object
        )
    else:
        data[target] = parse_dates(data[source])
cutoff = date(ANALYSIS_START_YEAR, 1, 1)
relevant = (
    data[["anchor_start", "anchor_end", "load_date", "ipl_date"]].isna().any(axis=1)
    | (data["anchor_end"] >= cutoff)
    | data["load_date"].ge(ANALYSIS_START)
    | data["ipl_date"].ge(ANALYSIS_START)
)
key = ["case_id", "anchor_start"]
valid_key = data["case_id"].fillna("").str.strip().ne("") & data["anchor_start"].notna()
history = relevant.groupby([data[column] for column in key], dropna=False).transform("max")
report_scope = relevant | (valid_key & history)
display(pd.Series({
    "Данные с 2024 и необходимая история": int(report_scope.sum()),
    "Исключены полностью старые записи": int((~report_scope).sum()),
}).to_frame("Строк"))
data = data.loc[report_scope].copy()
open_end = data["anchor_end"].eq(date(9999, 12, 31))
checks = {
    "Пустой case_id": data["case_id"].fillna("").str.strip().eq(""),
    "Некорректный Ankerbeginn": data["anchor_start"].isna(),
    "Некорректный Ankerende": data["anchor_end"].isna(),
    "Некорректный erstes Ladedatum": data["load_date"].isna(),
    "Некорректный IPL": data["ipl_date"].isna(),
    "Некорректные Lückentage": data["days"].isna() | ~np.isfinite(data["days"]),
    "Отрицательные Lückentage": data["days"].lt(0),
    "Дробные Lückentage": data["days"].notna() & data["days"].mod(1).ne(0),
    "Ankerende раньше Ankerbeginn": (data["anchor_end"] < data["anchor_start"].dt.date),
    "Первая загрузка раньше Ankerbeginn": data["load_date"].lt(data["anchor_start"]),
    "IPL раньше первой загрузки": data["ipl_date"].lt(data["load_date"]),
}
quality = pd.DataFrame({
    "Проверка": list(checks),
    "Строк": [int(mask.sum()) for mask in checks.values()],
})
display(quality)
display(data.groupby("sachverhalt", dropna=False).agg(
    rows=("case_id", "size"), first_load=("load_date", "min"),
    last_load=("load_date", "max"), last_ipl=("ipl_date", "max"),
).reset_index())
key = ["case_id", "anchor_start"]
valid = data.dropna(subset=["anchor_start", "anchor_end", "load_date", "ipl_date", "days"])
valid = valid.loc[valid["case_id"].fillna("").str.strip().ne("")].copy()
consistency = valid.groupby(key).agg(
    snapshots=("case_id", "size"),
    load_dates=("load_date", "nunique"),
    anchor_ends=("anchor_end", "nunique"),
    categories=("sachverhalt", "nunique"),
    day_values=("days", "nunique"),
)
display(pd.Series({
    "Уникальных ключей": len(consistency),
    "Ключей в нескольких строках": int(consistency["snapshots"].gt(1).sum()),
    "Разные даты первой загрузки": int(consistency["load_dates"].gt(1).sum()),
    "Разные даты окончания": int(consistency["anchor_ends"].gt(1).sum()),
    "Разные Sachverhalt": int(consistency["categories"].gt(1).sum()),
    "Изменялись Lückentage": int(consistency["day_values"].gt(1).sum()),
    "Повтор ключа в одном IPL": int(valid.duplicated(key + ["ipl_date"]).sum()),
}).to_frame("Количество"))


### Проверка нераспознанных дат Ankerende

Открытые периоды 31.12.9999 выводятся отдельно как информация и сохраняются
в анализе. В таблице нераспознанных дат остаются только пустые или ошибочные
значения. Примеры и частоты берутся из актуальной выборки RSA / 4.15.1 с необходимой историей, до исключения ошибочных дат.


In [ ]:
print(f"Открытых периодов Ankerende = 31.12.9999 (корректные записи): {int(open_end.sum()):,}")

invalid_end = data.loc[data["anchor_end"].isna()]
print(f"Нераспознанных дат Ankerende: {len(invalid_end):,}")
display(
    invalid_end[
        ["anchor_start_raw", "anchor_end_raw", "load_raw", "ipl_raw"]
    ].head(10)
)
display(
    invalid_end["anchor_end_raw"]
    .value_counts(dropna=False)
    .head(10)
    .to_frame("Количество")
)


### Ankerende раньше Ankerbeginn

Отдельная проверка распознанных дат: количество строк и первые 10 примеров, в которых окончание анкерного периода раньше его начала. Исходные значения показаны для проверки источника.


In [ ]:
reversed_periods = data.loc[
    data["anchor_start"].notna()
    & data["anchor_end"].notna()
    & (data["anchor_end"] < data["anchor_start"].dt.date)
]

print(f"Строк, где Ankerende раньше Ankerbeginn: {len(reversed_periods):,}")
display(
    reversed_periods[
        ["sachverhalt", "anchor_start_raw", "anchor_end_raw", "load_raw", "ipl_raw", "days"]
    ].head(10)
)


## 2. Когорта приложения и распределение значений

Отбор выполняется **до** дедупликации: период случая пересекает анкерный год,
затем применяется Sachverhalt. Для недельного ряда вызывается существующая функция
приложения. Отдельная детализация нужна для распределений. Ankerjahr начинается
с 2024; «all» включает периоды, пересекающие 2024 год или последующие годы.
Ранние даты первой загрузки сохраняются для полного накопленного итога.

При неоднозначном ключе с разными датами первой загрузки или несколькими значениями
в одном последнем IPL результат выбора может быть неоднозначен. Это требует проверки
источника. Notebook показывает такие случаи агрегированно.


In [ ]:
from pack.data_access import load_anchor_cohort_weekly_data

latest_year = max(
    ANALYSIS_START_YEAR,
    int(valid["anchor_start"].dt.year.max()) if not valid.empty else ANALYSIS_START_YEAR,
    int(data["load_date"].dt.year.max()) if data["load_date"].notna().any() else ANALYSIS_START_YEAR,
)
years = list(range(ANALYSIS_START_YEAR, latest_year + 1))
if not years:
    raise RuntimeError("Нет строк с пригодными датами.")
selected_year = ("all" if ANCHOR_YEAR == "all" else
                 int(ANCHOR_YEAR) if ANCHOR_YEAR is not None else max(years))
if selected_year != "all" and selected_year < ANALYSIS_START_YEAR:
    raise ValueError(f"Ankerjahr должен быть не раньше {ANALYSIS_START_YEAR}.")
selected_category = str(SACHVERHALT)

if selected_year == "all":
    selected = valid.loc[valid["anchor_end"].ge(cutoff)].copy()
else:
    selected = valid.loc[
        valid["anchor_start"].le(pd.Timestamp(selected_year, 12, 31))
        & valid["anchor_end"].ge(date(selected_year, 1, 1))
    ].copy()
if selected_category != "all":
    selected = selected.loc[selected["sachverhalt"].eq(selected_category)]
cases = selected.sort_values("ipl_date", kind="stable").drop_duplicates(key, keep="last")
display(pd.Series({
    "Ankerjahr": selected_year, "Sachverhalt": selected_category,
    "Строк снимков": len(selected), "Уникальных случаев": len(cases),
    "Первое Ladedatum": cases["load_date"].min(),
    "Последнее Ladedatum": cases["load_date"].max(),
}).to_frame("Выборка"))
if cases.empty:
    raise RuntimeError("Когорта пуста. Измените ANCHOR_YEAR / SACHVERHALT.")
display(cases["days"].describe(percentiles=[.01, .25, .5, .75, .95, .99]).to_frame("Lückentage / Fall"))
weekly_source = load_anchor_cohort_weekly_data(selected_category, selected_year)
if weekly_source.empty:
    raise RuntimeError("Функция приложения вернула пустую когорту.")
display(pd.Series({
    "Fälle из функции приложения": int(weekly_source["FAELLE"].sum()),
    "Fälle из детализации": len(cases),
    "Lückentage из функции приложения": weekly_source["TAGEN"].sum(),
    "Lückentage из детализации": cases["days"].sum(),
}).to_frame("Сверка"))

fig, ax = plt.subplots(figsize=(11, 5))
sns.histplot(data=cases, x="days", bins=60, ax=ax)
ax.set(title="Распределение Lückentage по случаям", xlabel="Lückentage", ylabel="Fälle")
show(fig)


## 3. Регулярная недельная сетка

Шаг — календарная неделя, понедельник–воскресенье. Индекс начинается с первой
недели появления данных когорты. Конец наблюдения по умолчанию — максимальное
erstes Ladedatum всей исследуемой выборки RSA / 4.15.1. Он является ориентиром,
пока полнота выгрузки не подтверждена.

Недельные графики и статистическая диагностика начинаются с 2024 года.
Накопленные итоги сохраняют всю необходимую историю когорты, включая ранние записи.

observed_* сохраняет NaN для недель без записей; new_* содержит нулевой прирост.
По подтверждённому пользователем правилу это обычные недели без новых случаев,
они не считаются дефектами и не разрывают диагностический ряд. Это правило
задаётся EMPTY_WEEKS_ARE_ZERO отдельно от подтверждения полноты выгрузки HANA.
Красная точка обозначает последнюю неделю с неполным покрытием
до выбранной даты конца наблюдения. Отсутствие семи дат загрузки внутри недели
само по себе не делает неделю неполной.


In [ ]:
if "weekly_source" not in globals():
    raise RuntimeError(
        "Сначала выполните ячейку раздела 2 «Когорта приложения и распределение значений» "
        "или выберите Restart Kernel → Run All. "
        "Если предыдущая ячейка завершилась ошибкой, устраните её и выполните заново."
    )

global_end = data["load_date"].max()
observation_end = pd.Timestamp(OBSERVATION_END) if OBSERVATION_END else global_end
if observation_end < pd.Timestamp(weekly_source["IPL_dt"].max()):
    raise ValueError("OBSERVATION_END раньше последней недели когорты.")
if observation_end < selected["load_date"].max():
    raise ValueError("OBSERVATION_END раньше последней фактической даты когорты.")

def regularize(source, end):
    observed = source.set_index("IPL_dt")[["FAELLE", "TAGEN"]].rename(
        columns={"FAELLE": "observed_cases", "TAGEN": "observed_days"}
    ).sort_index()
    end_week = end.normalize() - pd.Timedelta(days=end.weekday())
    index = pd.date_range(observed.index.min(), end_week, freq="W-MON", name="week")
    frame = observed.reindex(index)
    frame["missing_week"] = frame["observed_cases"].isna()
    frame["new_cases"] = frame["observed_cases"].fillna(0)
    frame["new_days"] = frame["observed_days"].fillna(0)
    frame["cumulative_cases"] = frame["new_cases"].cumsum()
    frame["cumulative_days"] = frame["new_days"].cumsum()
    frame["complete_calendar_week"] = frame.index + pd.Timedelta(days=6) <= end.normalize()
    return frame

weekly = regularize(weekly_source, observation_end)
display(weekly.tail(16))
assert weekly["new_cases"].sum() == weekly_source["FAELLE"].sum()
assert np.isclose(weekly["new_days"].sum(), weekly_source["TAGEN"].sum())
for metric, title in [("cases", "Fälle"), ("days", "Lückentage")]:
    fig, ax = plt.subplots(figsize=(13, 4))
    visible_weekly = weekly.loc[weekly.index >= ANALYSIS_START]
    sns.lineplot(data=visible_weekly, x=visible_weekly.index, y=f"new_{metric}", marker="o", ax=ax)
    missing = visible_weekly.loc[visible_weekly["missing_week"]]
    ax.scatter(missing.index, missing[f"new_{metric}"], color="C0", s=30,
               label="Нет новых случаев (0)", zorder=3)
    partial = visible_weekly.loc[~visible_weekly["complete_calendar_week"]]
    ax.scatter(partial.index, partial[f"new_{metric}"], color="red", s=65,
               label="Незавершённая неделя", zorder=4)
    ax.set(title=f"Недельные приросты {title} с {ANALYSIS_START_YEAR}",
           xlabel="Erstes Ladedatum / начало недели", ylabel=title)
    ax.legend()
    fig.autofmt_xdate()
    show(fig)

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
for ax, metric, title in zip(axes, ["cumulative_cases", "cumulative_days"], ["Fälle", "Lückentage"]):
    visible_weekly = weekly.loc[weekly.index >= ANALYSIS_START]
    sns.lineplot(data=visible_weekly, x=visible_weekly.index, y=metric, ax=ax)
    ax.set(title=f"Накопленные {title}", ylabel=title, xlabel="Erstes Ladedatum / начало недели")
fig.autofmt_xdate()
show(fig)


## 4. Стабильность, выбросы и связь показателей

Диагностики рассчитываются на приростах завершённых недель. Накопленные кривые
полезны для визуализации, но их корреляция может быть высокой только из-за
общего роста. Выбросы по IQR — кандидаты на изучение, они не удаляются.
Автокорреляция на лаге 52 осмысленна только при достаточно длинной истории.


In [ ]:
complete = weekly.loc[weekly["complete_calendar_week"] & (weekly.index >= ANALYSIS_START)].copy()
complete["days_per_case"] = complete["new_days"].div(complete["new_cases"].replace(0, np.nan))
diagnostics = []
for metric in ["new_cases", "new_days"]:
    series = complete[metric]
    q1, q3 = series.quantile([.25, .75]) if len(series) else (np.nan, np.nan)
    threshold = q3 + 1.5 * (q3 - q1)
    total = series.sum()
    diagnostics.append({
        "metric": metric, "weeks": len(series),
        "missing_weeks": int(complete["missing_week"].sum()),
        "zero_share": series.eq(0).mean(), "mean": series.mean(),
        "median": series.median(), "std": series.std(),
        "cv": series.std() / series.mean() if series.mean() else np.nan,
        "top_week_share": series.max() / total if total else np.nan,
        "iqr_upper_outliers": int(series.gt(threshold).sum()),
        "lag_1": series.autocorr(1) if len(series) >= 10 and series.nunique() > 1 else np.nan,
        "lag_52": series.autocorr(52) if len(series) >= 104 and series.nunique() > 1 else np.nan,
    })
display(pd.DataFrame(diagnostics))
display(complete[["new_cases", "new_days", "days_per_case"]].describe())
display(complete[["new_cases", "new_days"]].corr())
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=complete, x="new_cases", y="new_days", ax=ax)
ax.set(title="Связь недельных приростов", xlabel="Новые Fälle", ylabel="Новые Lückentage")
show(fig)
rolling = complete[["new_cases", "new_days"]].rolling(13, min_periods=4).mean()
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
for ax, metric in zip(axes, ["new_cases", "new_days"]):
    sns.lineplot(data=rolling, x=rolling.index, y=metric, ax=ax)
    ax.set(title=f"Средний прирост за 13 недель: {metric}", ylabel=metric, xlabel="Неделя")
fig.autofmt_xdate()
show(fig)
display(complete.nlargest(10, "new_days")[["new_cases", "new_days", "missing_week"]])


## 5. Сравнение когорт и пригодность отдельных рядов

Для каждой комбинации анкерного года и Sachverhalt строится отдельный ряд.
Пересекающие несколько лет случаи могут входить в несколько когорт, поэтому
суммировать их размеры между годами нельзя.

Практические пороги ниже служат предварительным фильтром: 52 недели обучения
и минимум один горизонт проверки. Для проверки годовой сезонности желательно
иметь несколько полных годовых циклов. Наличие нужного числа недель не доказывает
качество будущей модели. График относительного возраста выравнивает когорты
по окончанию анкерного года.


In [ ]:
# Ограничение числа лет можно изменить для большой базы.
comparison_years = years
categories = sorted(valid["sachverhalt"].dropna().unique())
summaries, maturity = [], []
for year in comparison_years:
    for category in categories:
        source = load_anchor_cohort_weekly_data(category, year)
        if source.empty:
            continue
        frame = regularize(source, observation_end)
        finished = frame.loc[frame["complete_calendar_week"] & (frame.index >= ANALYSIS_START)]
        usable = len(finished) if EMPTY_WEEKS_ARE_ZERO else int((~finished["missing_week"]).sum())
        summaries.append({
            "ankerjahr": year, "sachverhalt": category,
            "cases": int(source["FAELLE"].sum()), "days": source["TAGEN"].sum(),
            "calendar_weeks": len(finished), "weeks_with_records": int((~finished["missing_week"]).sum()),
            "missing_week_share": finished["missing_week"].mean(),
            "first_week": frame.index.min(), "last_record_week": source["IPL_dt"].max(),
            "enough_calendar_history": len(finished) >= MIN_TRAIN_WEEKS + HORIZON_WEEKS,
            "coverage_confirmed": CONFIRMED_COMPLETE_EXTRACT,
            "usable_weeks_under_assumption": usable,
        })
        if selected_category in ("all", category):
            path = frame.loc[frame.index >= ANALYSIS_START, ["cumulative_cases", "cumulative_days"]].copy()
            path["weeks_from_anchor_year_end"] = (
                (path.index - pd.Timestamp(year, 12, 31)).days / 7
            )
            path["cohort"] = f"{year} / {category}"
            maturity.append(path.reset_index())
summary = pd.DataFrame(summaries)
display(summary)
if maturity:
    aligned = pd.concat(maturity, ignore_index=True)
    for metric, title in [("cumulative_cases", "Fälle"), ("cumulative_days", "Lückentage")]:
        fig, ax = plt.subplots(figsize=(13, 6))
        sns.lineplot(data=aligned, x="weeks_from_anchor_year_end", y=metric,
                     hue="cohort", estimator=None, ax=ax)
        ax.set(title=f"Созревание когорт по {title}",
               xlabel="Недель от окончания анкерного года", ylabel=title)
        ax.legend(title="Когорта", bbox_to_anchor=(1.02, 1), loc="upper left")
        show(fig)


## 6. Автокорреляция и стационарность приростов

Исследуем регулярные недельные приросты, исключая незавершённую неделю.
Используем календарный ряд с 2024 года, включая обычные недели с нулевым
поступлением. Полнота выгрузки проверяется отдельно. Если изменить правило
EMPTY_WEEKS_ARE_ZERO на False, используем самый длинный непрерывный отрезок
с записями; удалять отдельные пропуски и склеивать оставшиеся недели нельзя.

ACF/PACF показывают зависимость по неделям. Полосы 95% точечные, множество лагов
увеличивает риск случайных пиков. ACF не доказывает предсказуемость.
ADF проверяет гипотезу единичного корня, KPSS — стационарности вокруг константы.
Оба теста чувствительны к длине ряда, выбросам и изменениям режима.


In [ ]:
import warnings
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import STL

def analysis_segment(frame):
    finished = frame.loc[frame["complete_calendar_week"] & (frame.index >= ANALYSIS_START)]
    if EMPTY_WEEKS_ARE_ZERO or finished.empty:
        return finished.copy()
    available = ~finished["missing_week"]
    groups = available.ne(available.shift()).cumsum()
    segments = [group for _, group in finished.loc[available].groupby(groups[available])]
    return max(segments, key=len).copy() if segments else finished.iloc[:0].copy()

analysis = analysis_segment(weekly)
display(pd.Series({
    "Завершённых календарных недель": len(complete),
    "Недель диагностического отрезка": len(analysis),
    "Начало": analysis.index.min(),
    "Конец": analysis.index.max(),
    "Нулевые недели — нормальные паузы": EMPTY_WEEKS_ARE_ZERO,
    "Полнота выгрузки подтверждена": CONFIRMED_COMPLETE_EXTRACT,
}).to_frame("Основа диагностики"))

def stationarity_tests(series):
    series = series.astype(float)
    result = {"n": len(series), "adf_p": np.nan, "kpss_p": np.nan, "status": ""}
    if len(series) < 24 or series.nunique() < 2:
        result["status"] = "Недостаточно наблюдений / постоянный ряд"
        return result
    messages = []
    for name, test in [
        ("adf", lambda: adfuller(series, regression="c", autolag="AIC")[1]),
        ("kpss", lambda: kpss(series, regression="c", nlags="auto")[1]),
    ]:
        try:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                result[f"{name}_p"] = test()
            messages.extend(str(w.message) for w in caught)
        except (ValueError, np.linalg.LinAlgError) as exc:
            messages.append(f"{name}: {exc}")
    adf_p, kpss_p = result["adf_p"], result["kpss_p"]
    if np.isfinite(adf_p) and np.isfinite(kpss_p):
        if adf_p < .05 and kpss_p >= .05:
            result["status"] = "Тесты согласуются со стационарностью"
        elif adf_p >= .05 and kpss_p < .05:
            result["status"] = "Тесты указывают на нестационарность"
        else:
            result["status"] = "Неоднозначно: проверить тренд, длину и сдвиги"
    result["warnings"] = " | ".join(messages)
    return result

test_rows, correlation_rows = [], []
for metric in ["new_cases", "new_days"]:
    for transform, series in [
        ("Прирост", analysis[metric]),
        ("Первая разность прироста", analysis[metric].diff().dropna()),
    ]:
        test_rows.append({"metric": metric, "transform": transform, **stationarity_tests(series)})
        if len(series) < 8 or series.nunique() < 2:
            continue
        lags = min(52, len(series) // 2 - 1)
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        plot_acf(series, lags=lags, zero=False, alpha=.05, ax=axes[0])
        plot_pacf(series, lags=lags, zero=False, alpha=.05, method="ywm", ax=axes[1])
        axes[0].set_title(f"ACF: {metric}, {transform}")
        axes[1].set_title(f"PACF: {metric}, {transform}")
        for ax in axes:
            ax.set_xlabel("Лаг, недели")
        show(fig)
    for lag in [1, 4, 13, 26, 52]:
        series = analysis[metric]
        correlation_rows.append({
            "metric": metric, "lag_weeks": lag,
            "acf": series.autocorr(lag) if len(series) >= max(24, 2 * lag) and series.nunique() > 1 else np.nan,
            "pairs": max(0, len(series) - lag),
        })
stationarity = pd.DataFrame(test_rows)
display(stationarity)
display(pd.DataFrame(correlation_rows))


## 7. Календарная сезонность и созревание

Профили и тепловая карта показывают различия по календарным годам. Общий профиль
может смешивать сезонность, созревание когорты и изменения процесса загрузки.
STL с периодом 52 — приближение годового цикла для недельной сетки.
Выполняем его при наличии хотя бы 156 последовательных недель; это практический
минимум для исследования, а не доказательство устойчивой сезонности.


In [ ]:
for metric in ["new_cases", "new_days"]:
    calendar = analysis[[metric]].copy()
    calendar["year"] = calendar.index.isocalendar().year.astype(int)
    calendar["week_of_year"] = calendar.index.isocalendar().week.astype(int)
    if not calendar.empty:
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        sns.lineplot(data=calendar, x="week_of_year", y=metric, hue="year",
                     palette="tab10", estimator=None, ax=axes[0])
        axes[0].set_title(f"Календарный профиль: {metric}")
        matrix = calendar.pivot_table(index="year", columns="week_of_year", values=metric)
        sns.heatmap(matrix, cmap="YlGnBu", ax=axes[1])
        axes[1].set_title("Пропуски остаются пустыми")
        show(fig)
    if len(analysis) >= 156 and analysis[metric].nunique() > 1:
        decomposition = STL(analysis[metric].astype(float), period=52, robust=True).fit()
        fig = decomposition.plot()
        fig.set_size_inches(13, 9)
        fig.suptitle(f"STL, период 52: {metric}", y=1.01)
        show(fig)
    else:
        print(f"{metric}: STL пропущен — нужно минимум 156 последовательных недель и непостоянный ряд.")


## 8. Изменение среднего, разброса и частоты нулей

Сравниваем последовательные блоки по 13 недель. Последний короткий блок показан
с фактическим размером. Изменение среднего не является автоматическим доказательством
смены режима: оно может отражать созревание когорты.


In [ ]:
stability_rows = []
for metric in ["new_cases", "new_days"]:
    for _, block in analysis.groupby(np.arange(len(analysis)) // 13):
        series = block[metric]
        stability_rows.append({
            "metric": metric, "start": block.index.min(), "end": block.index.max(),
            "weeks": len(block), "mean": series.mean(), "std": series.std(),
            "zero_share": series.eq(0).mean(), "max": series.max(),
            "days_per_case": block["new_days"].sum() / block["new_cases"].sum()
                if block["new_cases"].sum() else np.nan,
        })
stability = pd.DataFrame(stability_rows)
display(stability)
if not analysis.empty:
    fig, axes = plt.subplots(3, 2, figsize=(15, 11), sharex=True)
    for column, (metric, label) in enumerate([
        ("new_cases", "Fälle"), ("new_days", "Lückentage"),
    ]):
        series = analysis[metric]
        rolling = series.rolling(13, min_periods=13)
        statistics = [
            (rolling.mean(), "Среднее за 13 недель", label),
            (rolling.std(), "Стандартное отклонение за 13 недель", label),
            (series.eq(0).rolling(13, min_periods=13).mean(), "Доля нулей за 13 недель", "Доля"),
        ]
        for row, (values, title, unit) in enumerate(statistics):
            ax = axes[row, column]
            ax.plot(values.index, values, color=f"C{column}")
            ax.set(title=f"{title}: {label}", ylabel=unit)
            if row == 2:
                ax.set_ylim(-0.02, 1.02)
                ax.set_xlabel("Неделя")
    fig.autofmt_xdate()
    show(fig)


## 9. Заключение по каждой когорте

Сводка — основание для выбора экспериментов, а не автоматическое одобрение модели.
Тесты приводятся для приростов обеих величин. Кандидаты требуют проверки на
одинаковых горизонтах относительно простых baseline. Сезонные модели требуют
подтверждения повторяемости рисунка. Неподтверждённое покрытие блокирует окончательный
вывод о качестве. Графики выше относятся к выбранной когорте; сводка охватывает
все комбинации из раздела 5.


In [ ]:
assessment_rows = []
for year in comparison_years:
    for category in categories:
        source = load_anchor_cohort_weekly_data(category, year)
        if source.empty:
            continue
        frame = regularize(source, observation_end)
        segment = analysis_segment(frame)
        finished = frame.loc[frame["complete_calendar_week"] & (frame.index >= ANALYSIS_START)]
        for metric in ["new_cases", "new_days"]:
            series = segment[metric]
            tests = stationarity_tests(series)
            q1, q3 = series.quantile([.25, .75]) if len(series) else (np.nan, np.nan)
            outliers = int(series.gt(q3 + 1.5 * (q3 - q1)).sum())
            candidates = "Нулевой прирост; последнее значение; среднее/медиана окна"
            if len(series) >= 52 and series.nunique() > 1:
                candidates += "; проверить затухающий ETS и ARIMA"
            blockers = []
            if not CONFIRMED_COMPLETE_EXTRACT:
                blockers.append("Подтвердить полноту выгрузки контрольными показателями HANA")
            if not EMPTY_WEEKS_ARE_ZERO:
                blockers.append("Уточнить интерпретацию недель без записей")
            if len(series) < MIN_TRAIN_WEEKS + HORIZON_WEEKS:
                blockers.append("Мало последовательных недель для выбранного горизонта")
            if outliers:
                blockers.append("Объяснить недельные скачки")
            assessment_rows.append({
                "ankerjahr": year, "sachverhalt": category, "metric": metric,
                "calendar_weeks": len(finished), "diagnostic_weeks": len(series),
                "missing_week_share": finished["missing_week"].mean(),
                "zero_share_in_segment": series.eq(0).mean(),
                "iqr_upper_outliers": outliers,
                "acf_1": series.autocorr(1) if len(series) >= 24 and series.nunique() > 1 else np.nan,
                "acf_52": series.autocorr(52) if len(series) >= 104 and series.nunique() > 1 else np.nan,
                "adf_p": tests["adf_p"], "kpss_p": tests["kpss_p"],
                "stationarity": tests["status"], "test_warnings": tests.get("warnings", ""),
                "annual_cycles": round(len(series) / 52, 1),
                "seasonality_status": "Проверить календарные профили / STL" if len(series) >= 156 else "Мало циклов для устойчивого вывода",
                "candidate_models": candidates,
                "limitations": "; ".join(blockers) or "Нужна проверка прогноза на исторических срезах",
            })
assessment = pd.DataFrame(assessment_rows)
display(assessment)


## 10. Проверяемость будущего прогноза

Показываем возможные границы rolling-origin проверки на равномерной сетке.
Это календарный план, модель здесь не обучается. Последняя незавершённая неделя
исключена. Нулевые недели считаются обычными паузами по пояснению пользователя.
Полнота выгрузки этим не подтверждается.

Последние значения дней из IPL-снимков могли стать известны позже даты отсечения.
Для честного backtesting нужно восстановить состояние каждого случая по IPL,
доступному на тот момент, и проверить, что выгрузка содержит нужную историю снимков.
Ось целевого ряда при этом остаётся erstes Ladedatum.


In [ ]:
folds = []
for origin_index in range(MIN_TRAIN_WEEKS - 1, len(complete) - HORIZON_WEEKS, HORIZON_WEEKS):
    train = complete.iloc[:origin_index + 1]
    test = complete.iloc[origin_index + 1:origin_index + 1 + HORIZON_WEEKS]
    folds.append({
        "train_start": train.index.min(), "train_end": train.index.max(),
        "test_start": test.index.min(), "test_end": test.index.max(),
        "train_weeks": len(train), "test_weeks": len(test),
        "test_missing_weeks": int(test["missing_week"].sum()),
    })
display(pd.DataFrame(folds))
recommendations = [
    ("Даты и числовые значения", "Требует проверки" if quality["Строк"].sum() else "Проверки пройдены"),
    ("Недели без новых случаев", "Нормальные нулевые приросты" if EMPTY_WEEKS_ARE_ZERO else "Нужно уточнить"),
    ("Полнота выгрузки HANA", "Подтверждена" if CONFIRMED_COMPLETE_EXTRACT else "Нужен контроль количества и суммы из HANA"),
    ("Календарная глубина", "Достаточно для пробной проверки" if folds else "Недостаточно при выбранных train/horizon"),
    ("IPL и утечка будущих данных", "Проверить возможность восстановления исторических срезов"),
    ("Прогнозная точность", "Оценивается отдельно на backtesting"),
]
display(pd.DataFrame(recommendations, columns=["Область", "Вывод"]))
database_hash_after = hashlib.sha256(database.read_bytes()).hexdigest()
print("Файл базы не изменился за время исследования:", database_hash_after == database_hash_before)


## Вывод исследователя

После просмотра результатов заполните:

- Какие Ankerjahr × Sachverhalt имеют достаточно наблюдений?
- Как обычные недели с нулевым поступлением влияют на выбранный прогноз?
- Чем объясняются крупнейшие недельные скачки: поступление случаев или пакетная загрузка?
- Устойчиво ли число Lückentage на случай и нужен ли отдельный прогноз этого отношения?
- Есть ли повторяющаяся сезонность и достаточно ли полных циклов для её проверки?
- Какой горизонт нужен бизнесу и сколько исторических окон его покрывают?
- Можно ли восстановить прошлые срезы без более поздних изменений дней?
- Какие ограничения нужно устранить до сравнения моделей?

Следующий шаг после согласования качества данных — сравнение простых baseline
и выбранных моделей на одинаковых исторических окнах.
